# Etapa 5 — Evaluación y validación

**Modelo de Machine Learning para el cribado técnico de reservorios candidatos a waterflooding**

Proyecto de titulación — Maestría en Petróleos, ESPOL

---

| Paso | Contenido |
|---|---|
| 5.1 | Evaluación única sobre el conjunto de prueba (1 000 escenarios) |
| 5.2 | Comparación entrenamiento / validación cruzada / prueba |
| 5.3 | Matriz de confusión, precisión, recall, F1, exactitud y AUC-ROC |
| 5.4 | Importancia de variables y controles negativos |
| 5.5 | Comparación con el cribado tradicional por rangos |
| 5.6 | Caso de referencia con datos de campo (contexto, no validación) |

**Reglas de esta etapa.** Los hiperparámetros son los de la Etapa 4 y no se vuelve a ajustar nada.
El modelo final se elige por su F1 en validación cruzada, antes de mirar la prueba. El conjunto de
prueba se usa una sola vez, para informar el desempeño esperado.

## 1. Configuración del entorno

In [ ]:
import os, sys, subprocess, json

REPO_URL = "https://github.com/phabelog/waterflooding-ml-screening.git"
REPO_DIR = "waterflooding-ml-screening"

if os.path.isdir("src/evaluation"):
    REPO_ROOT = "."
else:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "-q", REPO_URL], check=True)
    else:
        subprocess.run(["git", "-C", REPO_DIR, "pull", "-q"], check=True)
    REPO_ROOT = REPO_DIR

sys.path.insert(0, os.path.join(REPO_ROOT, "src", "evaluation"))
print("Repositorio listo en:", os.path.abspath(REPO_ROOT))

In [ ]:
import sklearn, xgboost, pandas as pd
print("scikit-learn:", sklearn.__version__, "| xgboost:", xgboost.__version__)

import evaluate as ev, figures, train_models as tm

df = pd.read_csv(os.path.join(REPO_ROOT, "data", "synthetic", "synthetic_dataset.csv"))
e4 = tm.load_results(os.path.join(REPO_ROOT, "results", "etapa4_resultados.json"))
print("Escenarios:", len(df), "| modelos de la Etapa 4:", ", ".join(e4["modelos"]))

## 2. Evaluación completa

Ajusta los cuatro modelos con los 4 000 escenarios de entrenamiento y los evalúa. Tarda de 2 a 4 minutos.

In [ ]:
resultados, objetos = ev.run_evaluation(df, e4, n_jobs=-1)

## 3. Entrenamiento, validación cruzada y prueba

In [ ]:
tabla = objetos["tabla"].round(3)
tabla.index = tabla.index.set_levels([[tm.MODEL_LABELS[m] for m in tabla.index.levels[0]], tabla.index.levels[1]])
tabla

In [ ]:
print("Modelo final:", tm.MODEL_LABELS[resultados["modelo_final"]])
print("Matriz de confusión en la prueba:", resultados["modelos"][resultados["modelo_final"]]["matriz_confusion_prueba"])

## 4. Importancia de variables y controles negativos

In [ ]:
objetos["importancia"].round(4)

## 5. Comparación con el cribado tradicional por rangos

In [ ]:
fin = resultados["modelo_final"]
pd.DataFrame({"Clasificador trivial": resultados["clasificador_trivial_prueba"],
              "Cribado por rangos": {k: v for k, v in resultados["cribado_tradicional_prueba"].items() if k != "matriz_confusion"},
              tm.MODEL_LABELS[fin]: resultados["modelos"][fin]["prueba"]}).round(3)

## 6. Caso de referencia (datos de campo anonimizados; contexto, no validación)

In [ ]:
print(json.dumps(resultados["caso_referencia"], ensure_ascii=False, indent=2))
pd.DataFrame(ev.REFERENCE_TABLE, columns=["Variable", "Unidad", "Rango sintético", "Valor de campo", "Dentro del rango"])

## 7. Figuras y archivo de resultados

In [ ]:
from IPython.display import Image, display
fig_dir = os.path.join(REPO_ROOT, "docs", "figures")
rutas = figures.make_all(resultados, objetos, fig_dir)
ev.save_results(resultados, os.path.join(REPO_ROOT, "results", "etapa5_resultados.json"))
for r in rutas:
    display(Image(r, width=700))

## 8. Verificaciones automáticas

In [ ]:
r = subprocess.run([sys.executable, os.path.join(REPO_ROOT, "tests", "test_evaluation.py")], capture_output=True, text=True)
print(r.stdout[-3500:], r.stderr[-1500:])

## 9. Descargar los resultados

Genera `resultados_etapa5.zip` con el archivo de resultados y las figuras.

In [ ]:
import shutil
tmp = "resultados_etapa5"
shutil.rmtree(tmp, ignore_errors=True); os.makedirs(os.path.join(tmp, "docs", "figures")); os.makedirs(os.path.join(tmp, "results"))
shutil.copy(os.path.join(REPO_ROOT, "results", "etapa5_resultados.json"), os.path.join(tmp, "results"))
for r in rutas:
    shutil.copy(r, os.path.join(tmp, "docs", "figures"))
objetos["referencia"].to_csv(os.path.join(tmp, "results", "etapa5_caso_referencia.csv"), index=False)
shutil.make_archive(tmp, "zip", tmp)
try:
    from google.colab import files
    files.download(tmp + ".zip")
except ImportError:
    print("Archivo generado:", os.path.abspath(tmp + ".zip"))